In [16]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, chi2_contingency, mannwhitneyu
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [17]:
import kagglehub

PROXY = "http://localhost:2080"
NROWS = 500000

def get_dataset_with_proxy():
    previous_http = os.environ.get('HTTP_PROXY')
    previous_https = os.environ.get('HTTPS_PROXY')
    os.environ['HTTP_PROXY'] = PROXY
    os.environ['HTTPS_PROXY'] = PROXY
    path = kagglehub.dataset_download("wordsforthewise/lending-club")
    if previous_http is not None:
        os.environ['HTTP_PROXY'] = previous_http
    if previous_https is not None:
        os.environ['HTTPS_PROXY'] = previous_https
    return path

path = get_dataset_with_proxy()
accepted_df = pd.read_csv(os.path.join(path, 'accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv'), nrows=NROWS, low_memory=False)
accepted_df["is_accepted"] = True
rejected_df = pd.read_csv(os.path.join(path, 'rejected_2007_to_2018Q4.csv/rejected_2007_to_2018Q4.csv'), nrows=NROWS)
rejected_df["is_accepted"] = False
df = pd.concat([accepted_df, rejected_df], ignore_index=True)
df_accepted = df[df["is_accepted"] == True].copy()
print("Loaded", len(df_accepted), "accepted loans.")

/var/folders/0m/wjwgxmxs2xl1pdp5hqg8qgr00000gn/T/ipykernel_62036/141221338.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  accepted_df["is_accepted"] = True


Loaded 500000 accepted loans.


## 1. verification_status vs int_rate


In [18]:
d = df_accepted[['verification_status', 'int_rate']].dropna()
d = d[d['verification_status'].isin(['Source Verified', 'Not Verified', 'Verified'])]
groups = [d.loc[d['verification_status'] == g, 'int_rate'].values for g in d['verification_status'].unique()]
f_stat, p_val = f_oneway(*groups)
print("One-way ANOVA: verification_status vs int_rate")
print("F-statistic:", round(f_stat, 4))
print("p-value:", p_val)
d.groupby('verification_status')['int_rate'].agg(['mean', 'count'])

One-way ANOVA: verification_status vs int_rate
F-statistic: 19111.446
p-value: 0.0


,mean,count
verification_status,,
Not Verified,10.973470,143182
Source Verified,12.631977,215223
Verified,14.105097,141593


## home_ownership vs dti 


In [19]:
d = df_accepted[['home_ownership', 'dti']].dropna()
d = d[d['dti'] < 100]  
model = ols('dti ~ C(home_ownership)', data=d).fit()
anova = anova_lm(model, typ=2)
print(anova)
p_val = anova.loc['C(home_ownership)', 'PR(>F)']
print(p_val)

                         sum_sq        df          F        PR(>F)
C(home_ownership)  2.999474e+04       3.0  121.39777  1.345489e-78
Residual           4.113844e+07  499499.0        NaN           NaN
1.3454893250116496e-78


## 3. Association between home_ownership and grade


In [20]:
d = df_accepted[['home_ownership', 'grade']].dropna()
ct = pd.crosstab(d['home_ownership'], d['grade'])
chi2, p_val, dof, expected = chi2_contingency(ct)
print("Chi2:", round(chi2, 4), " dof:", dof)
print("p-value:", p_val)

Chi2: 3199.0769  dof: 18
p-value: 0.0


## Predict annual_inc


In [21]:
reg_df = df_accepted[['annual_inc', 'loan_amnt', 'dti', 'fico_range_low', 'revol_bal', 'open_acc', 'emp_length']].copy()
reg_df = reg_df.dropna()
reg_df = reg_df[(reg_df['annual_inc'] > 0) & (reg_df['annual_inc'] < 2e6)]
reg_df = reg_df[(reg_df['dti'] > 0) & (reg_df['dti'] < 60)]
emp_map = {k: i for i, k in enumerate(['< 1 year', '1 year', '2 years', '3 years', '4 years', '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years'])}
reg_df['emp_length_num'] = reg_df['emp_length'].map(emp_map).fillna(-1)
reg_df = reg_df.drop(columns=['emp_length'])
X = reg_df.drop(columns=['annual_inc'])
y = reg_df['annual_inc']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print("R2 (test):", round(r2_score(y_test, y_pred), 4))
print("RMSE (test):", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))
print("Mean", np.mean(y_test))
pd.Series(model.coef_, index=X.columns)

R2 (test): 0.3293
RMSE (test): 44249.78
Mean 78492.5798800816


loan_amnt            1.775385
dti              -1941.881561
fico_range_low      42.222742
revol_bal            0.642896
open_acc          1369.135458
emp_length_num     535.734565
dtype: float64

##  loan_amnt vs loan_status?


In [22]:
d = df_accepted[['loan_amnt', 'loan_status']].dropna()
d = d[d['loan_status'].isin(['Current', 'Charged Off'])]
g1 = d.loc[d['loan_status'] == 'Current', 'loan_amnt'].values
g2 = d.loc[d['loan_status'] == 'Charged Off', 'loan_amnt'].values
u_stat, p_val = mannwhitneyu(g1, g2, alternative='two-sided')
print("U-statistic:", round(u_stat, 4))
print("p-value:", p_val)
d.groupby('loan_status')['loan_amnt'].agg(['median', 'count'])

U-statistic: 4658829408.5
p-value: 0.0


,median,count
loan_status,,
Charged Off,15000.0,78824
Current,16000.0,104240
